# 08. Motor + chassis + battery: a multi-point Pareto front

A small electric vehicle is co-designed from three independent subsystems:

- a **motor** picked from a discrete catalog (each entry has its own (torque, mass, cost) tuple),
- a **chassis** whose mass and cost scale with the load it must carry,
- a **battery** sized to the mission energy.

The three subsystems are coupled cyclically: the chassis must support the motor and battery, the motor's torque is sized by the total moving mass (which includes the chassis), and so on. The `System` builder closes the loop.

Because the motor catalog has Pareto-incomparable entries (lighter and more expensive, or heavier and cheaper), the system-level Pareto front has multiple points: real engineering tradeoffs surfaced automatically.


## Imports

In [1]:
from codesign import (
    AlgebraicDP, CatalogDP, CatalogEntry, NamedProduct, Reals,
    System, minimize_cost, solve,
)

## Subsystem 1: a motor catalog

Each entry says "this motor can deliver up to X torque, and costs Y mass and Z dollars." Several entries are mutually incomparable on (mass, cost).


In [2]:
motor = CatalogDP(
    F=NamedProduct({"torque": Reals(unit="N*m")}),
    R=NamedProduct({"mass": Reals(unit="kg"), "cost": Reals(unit="USD")}),
    catalog=[
        CatalogEntry(name="Tiny",          provides={"torque": 2.0},  costs={"mass": 0.20, "cost": 30.0}),
        CatalogEntry(name="Light-Premium", provides={"torque": 8.0},  costs={"mass": 0.50, "cost": 200.0}),
        CatalogEntry(name="Mid-Standard",  provides={"torque": 8.0},  costs={"mass": 0.80, "cost": 120.0}),
        CatalogEntry(name="Heavy-Budget",  provides={"torque": 20.0}, costs={"mass": 1.50, "cost": 90.0}),
        CatalogEntry(name="Light-Pro",     provides={"torque": 20.0}, costs={"mass": 0.90, "cost": 350.0}),
        CatalogEntry(name="XL-Budget",     provides={"torque": 80.0}, costs={"mass": 3.50, "cost": 180.0}),
        CatalogEntry(name="XL-Pro",        provides={"torque": 80.0}, costs={"mass": 2.20, "cost": 700.0}),
    ],
    name="motor",
)
motor

DP(motor: torque:R+[N*m] -> A[mass:R+[kg]×cost:R+[USD]])

## Subsystem 2: the chassis

In [3]:
chassis = AlgebraicDP(
    F=NamedProduct({"load": Reals(unit="kg")}),
    R=NamedProduct({"mass": Reals(unit="kg"), "cost": Reals(unit="USD")}),
    equations={
        "mass": lambda f: 0.6 * f["load"],
        "cost": lambda f: 20.0 * f["load"],
    },
    name="chassis",
)

## Subsystem 3: the battery

In [4]:
battery = AlgebraicDP(
    F=NamedProduct({"energy": Reals(unit="J")}),
    R=NamedProduct({"mass": Reals(unit="kg"), "cost": Reals(unit="USD")}),
    equations={
        "mass": lambda f: f["energy"] / 1.8e6,
        "cost": lambda f: 0.05 * f["energy"] / 3.6e3,  # $0.05 / Wh
    },
    name="battery",
)

## Wire it up

The chassis must support payload + motor + battery, the motor's torque demand depends on the total moving mass, and the battery's energy demand is the externally-supplied mission energy. The total mass and total cost aggregate up from all subsystems.


In [5]:
G = 9.81
TORQUE_PER_KG = 0.25

sys = System("vehicle")
sys.provides("payload", unit="kg")
sys.provides("mission_energy", unit="J")
sys.requires("total_mass", unit="kg")
sys.requires("total_cost", unit="USD")

sys.add("motor", motor)
sys.add("chassis", chassis)
sys.add("battery", battery)

sys.constrain("chassis.load",
              lambda x: x["payload"] + x["motor.mass"] + x["battery.mass"])
sys.constrain("motor.torque",
              lambda x: TORQUE_PER_KG * G * (
                  x["payload"] + x["chassis.mass"] + x["battery.mass"]))
sys.constrain("battery.energy",
              lambda x: x["mission_energy"])
sys.constrain("total_mass",
              lambda x: x["payload"] + x["motor.mass"]
                        + x["chassis.mass"] + x["battery.mass"])
sys.constrain("total_cost",
              lambda x: x["motor.cost"] + x["chassis.cost"] + x["battery.cost"])

vehicle = sys.build()
print(sys)

System('vehicle'):
  provides:
    payload: R+[kg]
    mission_energy: R+[J]
  requires:
    total_mass: R+[kg]
    total_cost: R+[USD]
  subsystems:
    motor: (torque) -> (mass, cost)
    chassis: (load) -> (mass, cost)
    battery: (energy) -> (mass, cost)
  constraints:
    chassis.load >= <demand>
    motor.torque >= <demand>
    battery.energy >= <demand>
    total_mass >= <demand>
    total_cost >= <demand>


## Solve for three mission profiles

In [6]:
cases = [
    ("Small parcel", dict(payload=2.0,  mission_energy=2.0e5)),
    ("Medium load",  dict(payload=10.0, mission_energy=1.0e6)),
    ("Heavy + long", dict(payload=20.0, mission_energy=5.0e6)),
]
for label, f in cases:
    result = solve(vehicle, f, max_iter=200)
    f_str = ", ".join(f"{k}={v}" for k, v in f.items())
    print(f"{label}: {f_str}")
    print(f"   iters={result.iterations}, feasible={result.feasible}")
    if not result.feasible:
        print()
        continue
    print(f"   Pareto front ({len(result.antichain.points)} points):")
    for p in result.antichain.points:
        print(f"      total_mass={p['total_mass']:6.2f} kg,  "
              f"total_cost=${p['total_cost']:7.2f}")
    cheapest = minimize_cost(result, cost_fn=lambda r: r["total_cost"])
    if cheapest is not None:
        print(f"   cheapest: total_mass={cheapest['total_mass']:.2f} kg, "
              f"total_cost=${cheapest['total_cost']:.2f}")
    print()

Small parcel: payload=2.0, mission_energy=200000.0
   iters=4, feasible=True
   Pareto front (2 points):
      total_mass=  4.82 kg,  total_cost=$ 413.00
      total_mass=  5.78 kg,  total_cost=$ 165.00
   cheapest: total_mass=5.78 kg, total_cost=$165.00

Medium load: payload=10.0, mission_energy=1000000.0
   iters=3, feasible=True
   Pareto front (2 points):
      total_mass= 22.49 kg,  total_cost=$ 475.00
      total_mass= 20.41 kg,  total_cost=$ 969.00
   cheapest: total_mass=22.49 kg, total_cost=$475.00

Heavy + long: payload=20.0, mission_energy=5000000.0
   iters=4, feasible=False



## Reading the Pareto fronts

For Small parcel and Medium load, two motor choices remain feasible after the chassis grows to support everything, and they trade total mass against total cost: one is lighter but more expensive, the other heavier but cheaper. Neither dominates, so both appear in the system Pareto front.

For Heavy + long, even the biggest motor in the catalog can't supply enough torque once the chassis (which scales with battery mass) grows large enough. The iteration drives the loop axis to `⊤` and the result is `feasible=False`, automatically and without any special handling in the example code.

This is the payoff of modularity: each subsystem has a clean local definition, and the global Pareto structure is discovered by the solver from the constraint graph alone.
